# Treino experimental da voz da Lia (Piper Plus / VITS)

**Notebook preparado para execução manual no Google Colab.** Ao executar o passo de upload, os arquivos de voz serão transferidos do seu computador para o runtime Google. Você autorizou esse envio nesta conversa. O notebook não monta Drive, não publica no Hugging Face e não envia áudio para terceiros além do ambiente Colab escolhido. O runtime é temporário; interrompa e apague a sessão ao terminar.

## Condição de licença

Candidato técnico: `ayousanz/piper-plus-base` (Piper Plus multilingual, inclui português). O model card publica dados/receita, mas não declara claramente a licença dos pesos. O código do Piper Plus se declara MIT, o que **não licencia os pesos**. A autorização dada nesta conversa permite a prova privada/local; o fluxo prossegue sem edição de código. Não publique nem redistribua o checkpoint/base/fine-tune sem confirmar os direitos.

Não use áudio de outra pessoa sem autorização. Revise transcrições manualmente. Este fluxo não faz ASR. Mantenha originais privados fora do Git.

**Entrada:** um ZIP preparado com `metadata.csv` no formato `id|texto` e arquivos `.wav` mono correspondentes. Exemplo de linha: `fala_001|Olá, esta é uma frase de teste.` O nome do arquivo deve ser `fala_001.wav`. Os exemplos devem ser falas naturais, sem cortes de fonemas. O notebook valida automaticamente se tudo corresponde; transcrições devem estar corretas. Separe frases inéditas para avaliação antes de treinar.

In [ ]:
# Controles explícitos. Nenhum upload ocorre nesta célula.
PRIVATE_EVALUATION_ONLY = True
ALLOW_UNRESOLVED_BASE_WEIGHTS_FOR_PRIVATE_TEST = True  # autorização do usuário: avaliar em privado; sem redistribuição

if not PRIVATE_EVALUATION_ONLY or not ALLOW_UNRESOLVED_BASE_WEIGHTS_FOR_PRIVATE_TEST:
    raise RuntimeError('Este fluxo foi autorizado apenas para avaliação privada, sem redistribuição.')
print('Modo privado autorizado. Nenhum áudio foi enviado ainda; o upload ocorre na célula própria.')

## 1. Ambiente do treino

Instalações ocorrem apenas dentro do Colab, não no Windows nem no repositório. Código fixado no commit consultado em 2026-09-30. **Antes de executar, selecione GPU no menu `Runtime > Change runtime type > Hardware accelerator > GPU`**; a sessão que gerou o erro estava sem GPU. O notebook não consegue mudar hardware por código. Depois use `Runtime > Run all`: ele instala o ambiente, pede o ZIP uma vez, valida os dados, treina, exporta e oferece o resultado para download.

In [ ]:
import pathlib, subprocess, torch
# Usa a API do PyTorch; não depende do executável nvidia-smi.
if not torch.cuda.is_available():
    raise RuntimeError('Este runtime Colab não tem GPU disponível. Antes de executar novamente, selecione Runtime > Change runtime type > Hardware accelerator > GPU e reinicie a sessão.')
print('GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
REPO = pathlib.Path('/content/piper-plus')
PIN = '03f22b895d6b87d3733223c0844cb2ef6569937b'
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/ayutaz/piper-plus.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'checkout',PIN], check=True)
subprocess.run(['python','-m','pip','install','uv'], check=True)
subprocess.run(['uv','pip','install','--system','.[train]'], cwd=REPO, check=True)
subprocess.run(['python','-m','pip','install','huggingface_hub'], check=True)
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())


## 2. Upload explícito do dataset

Ao executar a célula, o Colab abre o seletor para você escolher um único ZIP do computador; esse é o único passo de upload. O ZIP e as transcrições ficam temporariamente no runtime. Não inclua arquivos sem autorização de uso.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil
uploaded = files.upload()  # abre seletor local; envia somente o ZIP escolhido
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Envie exatamente um ZIP com metadata.csv e WAVs mono.')
DATA_ROOT = Path('/content/lia_voice_data')
shutil.rmtree(DATA_ROOT, ignore_errors=True)
DATA_ROOT.mkdir(parents=True)
with zipfile.ZipFile(zip_names[0]) as zf:
    root = DATA_ROOT.resolve()
    for info in zf.infolist():
        target = (DATA_ROOT / info.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError('Caminho inseguro dentro do ZIP; extração bloqueada.')
    zf.extractall(DATA_ROOT)
print('Extraído no armazenamento temporário:', DATA_ROOT)

## 3. Validar estrutura e áudio

`metadata.csv` usa pipe (`id|texto`), sem cabeçalho. Cada identificador precisa corresponder a `id.wav`. O pré-processamento Piper converte para 22.050 Hz. Revise transcrições, nomes, números, clipping, ruído e cortes antes do treino.

In [ ]:
import soundfile as sf
DATA_ROOT = Path('/content/lia_voice_data')
meta_files = list(DATA_ROOT.rglob('metadata.csv'))
if len(meta_files) != 1:
    raise ValueError(f'Esperado um metadata.csv; encontrados {len(meta_files)}')
META = meta_files[0]
AUDIO_DIR = META.parent
rows, missing, durations, rates = [], [], [], set()
for line_no, line in enumerate(META.read_text(encoding='utf-8').splitlines(), 1):
    if not line.strip() or line.lstrip().startswith('#'): continue
    parts = line.split('|', 1)
    if len(parts) != 2 or not all(part.strip() for part in parts):
        raise ValueError(f'Linha {line_no} inválida; esperado id|texto')
    key, text = parts[0].strip(), parts[1].strip()
    wav = AUDIO_DIR / (key + '.wav')
    if not wav.is_file():
        missing.append((line_no, key)); continue
    info = sf.info(wav)
    if info.channels != 1 or info.frames == 0:
        raise ValueError(f'{wav.name}: esperado WAV mono não vazio')
    rows.append((key, text)); durations.append(info.duration); rates.add(info.samplerate)
if missing: raise FileNotFoundError(f'Áudios ausentes: {missing[:10]}')
if not rows: raise ValueError('Dataset vazio')
print(f'{len(rows)} frases; {sum(durations)/60:.1f} min; duração {min(durations):.2f}-{max(durations):.2f} s')
print('Sample rates encontrados:', sorted(rates))
print('Validação estrutural concluída; faça revisão auditiva e transcrição antes de avançar.')

## 4. Localizar base, revisar licença e preparar

O notebook baixa automaticamente o checkpoint e `config.json` da revisão pinada do model card. Verifica SHA-256 e tamanho do checkpoint. O arquivo usa formato PyTorch checkpoint/pickle: só confie no artefato e no código upstream fixados. A licença do peso não está clara; conforme sua autorização, o uso fica restrito a avaliação privada, sem publicação ou redistribuição.

In [ ]:
from huggingface_hub import hf_hub_download
import hashlib
BASE_REPO = 'ayousanz/piper-plus-base'
BASE_REVISION = 'ce006e5da7851fdc469887026eeb3f2175e7e080'
BASE_CHECKPOINT = hf_hub_download(BASE_REPO, 'model.ckpt', revision=BASE_REVISION)
BASE_CONFIG = hf_hub_download(BASE_REPO, 'config.json', revision=BASE_REVISION)
EXPECTED_SHA256 = '83c079abc50575eeb837e1d21e98aefafb23981b9f2cbee247cbcd222df6216b'
sha = hashlib.sha256(Path(BASE_CHECKPOINT).read_bytes()).hexdigest()
if sha != EXPECTED_SHA256:
    raise ValueError(f'Hash inesperado do checkpoint: {sha}')
DATASET_DIR = '/content/lia_dataset'
OUT_DIR = '/content/lia_voice_experiment'
print('Checkpoint verificado:', sha, '| bytes:', Path(BASE_CHECKPOINT).stat().st_size)
preprocess_cmd = [
    'python','-m','piper_train.preprocess',
    '--input-dir',str(AUDIO_DIR),'--output-dir',DATASET_DIR,
    '--language','pt','--dataset-format','ljspeech','--sample-rate','22050',
    '--single-speaker','--phoneme-type','multilingual'
]
print('Pré-processamento proposto:', ' '.join(preprocess_cmd))
RUN_PREPROCESS = True
if RUN_PREPROCESS:
    subprocess.run(preprocess_cmd, cwd=REPO, check=True)
else:
    print('Pré-processamento concluído automaticamente.')

## 5. Treino

A receita segue os parâmetros publicados para Piper Plus multilingual, com 500 épocas e checkpoints de retomada a cada 25 épocas. O treino inicia automaticamente após upload e validações. A duração depende do tamanho do corpus e da GPU alocada; o Colab pode interromper sessões gratuitas.

In [ ]:
train_cmd = [
    'python','-m','piper_train','--dataset-dir',DATASET_DIR,
    '--prosody-dim','16','--accelerator','gpu','--devices','1',
    '--precision','32-true','--max_epochs','500','--batch-size','4',
    '--samples-per-speaker','4','--checkpoint-epochs','25',
    '--base_lr','2e-5','--disable_auto_lr_scaling','--ema-decay','0.9995',
    '--max-phoneme-ids','400','--no-wavlm',
    '--resume-from-multispeaker-checkpoint',BASE_CHECKPOINT,
    '--default_root_dir',OUT_DIR
]
print('Comando piloto:', ' '.join(train_cmd))
RUN_TRAINING = True
if RUN_TRAINING:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Este Colab está sem GPU NVIDIA. Selecione Runtime > Change runtime type > GPU, reconecte e execute as células novamente.')
    subprocess.run(train_cmd, cwd=REPO, check=True)
else:
    print('Treino não iniciado.')

## 6. Exportar, empacotar e baixar

Depois que o treino terminar, a próxima célula encontra o checkpoint mais recente, exporta ONNX, inclui o JSON de configuração e abre o download do pacote. Nenhum arquivo é enviado/publicado automaticamente. Guarde o ZIP fora do repositório; a licença dos pesos segue sem esclarecimento, então o uso continua privado.

Ao terminar, interrompa e exclua o runtime Colab. O notebook não monta Drive.


In [ ]:
from pathlib import Path
import shutil, subprocess, zipfile
from google.colab import files

ckpts = [p for p in Path(OUT_DIR).rglob('*.ckpt') if p.is_file()]
if not ckpts:
    raise FileNotFoundError('Treino finalizado, mas não encontrei checkpoint .ckpt em '+OUT_DIR)
latest = max(ckpts, key=lambda p: p.stat().st_mtime)
EXPORT_DIR = Path(OUT_DIR) / 'export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
ONNX = EXPORT_DIR / 'lia_voice.onnx'
export_cmd = ['python','-m','piper_train.export_onnx','--unify-emb-lang',str(latest),str(ONNX)]
print('Checkpoint selecionado:', latest)
subprocess.run(export_cmd, cwd=REPO, check=True)

# O exportador normalmente cria o JSON pareado. Se não o fizer, usar o config
# da mesma base pinada, cujo inventário de fonemas/arquitetura foi validado.
config_candidates = [Path(str(ONNX)+'.json'), ONNX.with_suffix('.onnx.json'), EXPORT_DIR/'config.json']
config_out = next((x for x in config_candidates if x.is_file()), None)
if config_out is None:
    config_out = Path(str(ONNX)+'.json')
    shutil.copy2(BASE_CONFIG, config_out)

package = Path('/content/lia_voice_private_eval.zip')
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX, ONNX.name)
    zf.write(config_out, 'lia_voice.onnx.json')
print('Pacote pronto:', package, '| bytes:', package.stat().st_size)
files.download(str(package))
